# Notebook 04 — Claim Contextualization
**CliniVer** · Experiment `EXP-04-001`

| | |
|---|---|
| **Objective** | Decide how a claim is turned into a retrieval query: alone, or together with the user's question and/or the model's answer, or rewritten into a standalone claim |
| **Research question** | **RQ3:** Does including the original query, and in what form, improve evidence retrieval? For which claims? |
| **Hypotheses (pre-registered)** | **H3a:** Query + claim retrieval improves Recall@10 over claim-only for **context-dependent** claims. **H3b:** For **self-contained** claims, adding the query does not improve, and may degrade, precision |
| **Decision rules (fixed before running, `configs/context.yaml`)** | **H3a** supported if Recall@10(B) − Recall@10(A) > 0 (95% CI excludes 0) on context-dependent claims, for **both** retrievers. **H3b** supported if nDCG@10(B) − nDCG@10(A) is **not** significantly positive on self-contained claims, for both retrievers. The pipeline strategy is the one with the highest mean nDCG@10 (both strata, both retrievers) on the **tuning** split, reported on the **evaluation** split |

> **Synthetic data, stated up front.** No public dataset combines questions, context-dependent claims and evidence relevance labels. The context-dependent test items are therefore **constructed** from real SciFact claims with the local LLM and validated by rules. Their relevant documents are the expert evidence labels of the original claims. Results on them are labelled synthetic (research_foundation §10.2).

## 1. Background and method
### 1.1 Strategies
| ID | Query | Notes |
|---|---|---|
| A | claim | baseline |
| B | question + claim | concatenated |
| C | question + claim + answer | "M" in the design is interpreted as the model's **answer** (response) |
| D | question and claim retrieved separately, merged by reciprocal-rank fusion | k = 60 |
| E | weighted score fusion of the two sub-queries | claim weight tuned on the tuning split |
| F | LLM rewrite of the claim into a standalone claim (molecular-facts style), then retrieval with the rewrite | Qwen3-8B, schema-constrained |
| oracle | the original SciFact claim | upper bound for the context-dependent stratum |

### 1.2 Test items
- **Self-contained stratum:** the 693 SciFact train/dev claims that have evidence (expert-written, standalone by design), each paired with a generated question and answer.
- **Context-dependent stratum:** for each claim, Qwen3-8B writes a question that names the claim's main subject without revealing the finding, a 2–3 sentence answer, and a one-sentence version of the claim that refers back to the question ("It reduces…") instead of naming the subject.
- **Validation rules** (`cliniver.claims.contextualize.validate_context`):
  - the removed entity occurs in the claim and in the question, and not in the dependent claim;
  - one sentence, of similar length;
  - ≥ 70% of its content words come from the original claim;
  - the question does not restate the claim.

  Invalid items are excluded and counted.
- **Corpus:** the 5,183 SciFact abstracts (title + abstract, indexed whole). Relevant = documents with SUPPORT or CONTRADICT evidence.

### 1.3 Real claims (NB03 output)
- The 2,777 claims extracted from real generated answers are tagged as context-dependent or not by a rule tagger (unresolved referent, or a key domain entity that appears only in the question), with a manual spot check.
- Strategy F is applied to the context-dependent ones (and to 100 self-contained controls, which should come back unchanged).
- There are no relevance labels for real claims, so this part measures **prevalence** and **rewrite behaviour**, not retrieval quality.

### 1.4 Metrics and statistics
- Recall@10, P@5, MRR, nDCG@10; **query drift** = 1 − cos(query, original claim) with the dense encoder.
- Paired bootstrap over claims (2,000 resamples).

## 2. Setup

In [ ]:
import json, os, random, re, time, collections
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, yaml
from IPython.display import display

from cliniver.config import load_project_config
from cliniver.config.llm_config import load_llm_config
from cliniver.config.processing_config import load_processing_config
from cliniver.utils.seed import set_seed
from cliniver.utils.experiment import Experiment
from cliniver.ingestion.http import RawStore, RateLimitedClient, cached_get
from cliniver.ingestion.scifact import parse_release
from cliniver.ingestion.qa_datasets import domain_regex
from cliniver.llm.runtime import ManagedClient
from cliniver.claims.contextualize import make_context, decontextualize, context_dependent, _REFERENT
from cliniver.retrieval.simple import BM25Index, DenseIndex
from cliniver.retrieval.fusion import rrf, weighted
from cliniver.evaluation.retrieval import doc_metrics
from cliniver.evaluation.span_retrieval import cluster_bootstrap_diff

pcfg = load_project_config(); lcfg = load_llm_config(); prc = load_processing_config()
cfg = yaml.safe_load((pcfg.paths.absolute("data_raw").parents[1] / "configs" / "context.yaml").read_text(encoding="utf-8"))
SEED = set_seed(cfg["seed"]); SNAP = cfg["snapshot"]
SMOKE = os.environ.get("CLINIVER_SMOKE") == "1"   # code check on small subsets; shares the LLM call cache; never results
RAW = pcfg.paths.absolute("data_raw"); PROC = pcfg.paths.absolute("data_processed") / SNAP; ANN = RAW.parent / "annotations"
LLM_LOG = pcfg.paths.absolute("data_interim") / SNAP / "llm_calls"
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 140)
exp = Experiment(exp_id="EXP-04-000-smoke" if SMOKE else "EXP-04-001", title="Claim contextualization", research_question="RQ3 / H3a, H3b",
                 config={"context": cfg, "llm_model": lcfg.models[cfg["llm_model"]].model_dump(), "dense": prc.retriever.model_dump()},
                 seed=SEED, packages=["torch", "sentence-transformers", "bm25s", "pandas", "cliniver"])
def managed():
    return ManagedClient(lcfg, cfg["llm_model"], LLM_LOG / f"{cfg['llm_model']}.jsonl", server_log=pcfg.paths.absolute("logs") / "llama-server_nb04.log")
print("seed", SEED, "| LLM", cfg["llm_model"], "| dense", prc.retriever.dense_model)

## 3. Data: SciFact claims, corpus, and synthetic context

In [ ]:
raw = cached_get(RateLimitedClient("scifact", min_interval_s=1.0, max_in_flight=1), RawStore(RAW, "scifact", SNAP), "data", "tar.gz", cfg["scifact"]["url"])
corpus, claims = parse_release(raw)
if SMOKE:
    claims = claims[:20]
doc_ids = list(corpus); doc_text = [corpus[d] for d in doc_ids]
ids_sorted = sorted(c.claim_id for c in claims)
rng = random.Random(SEED); rng.shuffle(ids_sorted)
n_tune = round(len(ids_sorted) * cfg["scifact"]["tuning_fraction"])
split_of = {cid: ("tuning" if i < n_tune else "evaluation") for i, cid in enumerate(ids_sorted)}
print(f"corpus {len(corpus)} abstracts | claims with evidence {len(claims)} | relevant docs per claim mean {np.mean([len(c.relevant) for c in claims]):.2f}")

In [ ]:
items, invalid = [], collections.Counter()
t0 = time.perf_counter()
with managed() as mc:
    for c in claims:
        it, rec = make_context(mc, c.text, seed=SEED, tag=f"ctx:{c.claim_id}")
        if it is None:
            invalid["invalid JSON"] += 1; continue
        if not it.valid:
            invalid[it.reason] += 1; continue
        items.append({"claim_id": c.claim_id, "split": split_of[c.claim_id], "claim": c.text, "question": it.question, "answer": it.answer,
                      "dependent": it.dependent_claim, "removed_entity": it.removed_entity, "relevant": sorted(c.relevant)})
    gen_stats = mc.stats
print(f"generation {time.perf_counter()-t0:.0f}s; server: {gen_stats}")
synth = {"claims": len(claims), "valid_items": len(items), "invalid": dict(invalid),
         "valid_rate": round(len(items) / len(claims), 3), "by_split": dict(collections.Counter(i["split"] for i in items))}
exp.log_result("synthetic_items", synth); print(json.dumps(synth, indent=1))
display(pd.DataFrame(items)[["claim", "question", "dependent"]].sample(6, random_state=SEED))

### 3.1 Strategy F rewrites (both strata)
The same rewrite prompt is applied to the dependent claim (it should restore the subject) and to the original claim (it should return it essentially unchanged).

In [ ]:
with managed() as mc:
    for it in items:
        for key in ("dependent", "claim"):
            out, _ = decontextualize(mc, it[key], it["question"], it["answer"], seed=SEED, tag=f"dectx:{key}:{it['claim_id']}")
            it[f"F_{key}"] = out or it[key]
            it[f"F_{key}_ok"] = out is not None
ent_restored = np.mean([it["removed_entity"].lower() in it["F_dependent"].lower() for it in items])
unchanged = np.mean([re.sub(r"\W+", " ", it["F_claim"].lower()).strip() == re.sub(r"\W+", " ", it["claim"].lower()).strip() for it in items])
fstats = {"valid_json": float(np.mean([it["F_dependent_ok"] and it["F_claim_ok"] for it in items])),
          "dependent_entity_restored": round(float(ent_restored), 3), "self_contained_returned_unchanged": round(float(unchanged), 3)}
exp.log_result("rewrite_F", fstats); print(fstats)

## 4. Retrieval experiments (A–F, oracle) × 2 strata × 2 retrievers

In [ ]:
from sentence_transformers import SentenceTransformer
enc = SentenceTransformer(prc.retriever.dense_model, revision=prc.retriever.dense_revision, device="cuda" if torch.cuda.is_available() else "cpu")
bm = BM25Index(doc_text)
de = DenseIndex(doc_text, enc, batch_size=prc.retriever.batch_size, query_prefix=prc.retriever.query_prefix)
DEPTH = cfg["candidate_depth"]
def search(index, queries):
    r = index.search(queries, DEPTH)
    return r.ids, r.scores

STRATA = {"self-contained": "claim", "context-dependent": "dependent"}
runs = {}
for rname, idx in (("bm25", bm), ("dense", de)):
    q_ids, q_sc = search(idx, [it["question"] for it in items])
    for stratum, key in STRATA.items():
        C = [it[key] for it in items]
        c_ids, c_sc = search(idx, C)
        runs[(rname, stratum, "A")] = c_ids
        runs[(rname, stratum, "B")] = search(idx, [f"{it['question']} {c}" for it, c in zip(items, C)])[0]
        runs[(rname, stratum, "C")] = search(idx, [f"{it['question']} {c} {it['answer']}" for it, c in zip(items, C)])[0]
        runs[(rname, stratum, "D")] = [rrf([c_ids[i], q_ids[i]], k=cfg["rrf_k"], top=100) for i in range(len(items))]
        for w in cfg["fusion_weights"]:
            runs[(rname, stratum, f"E{w}")] = [weighted(c_ids[i], c_sc[i], q_ids[i], q_sc[i], w) for i in range(len(items))]
        runs[(rname, stratum, "F")] = search(idx, [it[f"F_{key}"] for it in items])[0]
        if stratum == "context-dependent":
            runs[(rname, stratum, "oracle")] = search(idx, [it["claim"] for it in items])[0]
rows = []
for (rname, stratum, strat), ranked in runs.items():
    for it, r in zip(items, ranked):
        m = doc_metrics([doc_ids[j] for j in r[:100]], set(it["relevant"]))
        rows.append({"retriever": rname, "stratum": stratum, "strategy": strat, "claim_id": it["claim_id"], "split": it["split"], **m})
R = pd.DataFrame(rows)
exp.save_table("per_claim_metrics", R)
# strategy E: choose the claim weight on the tuning split (mean nDCG@10 over strata and retrievers)
e_tune = R[(R.split == "tuning") & R.strategy.str.startswith("E")].groupby("strategy")["ndcg@10"].mean()
E_BEST = e_tune.idxmax()
R = R[~R.strategy.str.startswith("E") | (R.strategy == E_BEST)].replace({"strategy": {E_BEST: "E"}})
exp.log_result("fusion_weight", {"chosen": E_BEST, "tuning_ndcg": e_tune.round(4).to_dict()})
print("strategy E weight:", E_BEST)

In [ ]:
METRICS = ["recall@10", "p@5", "mrr", "ndcg@10"]
summary = R.groupby(["split", "stratum", "retriever", "strategy"])[METRICS].mean().round(3)
evals = summary.loc["evaluation"]
exp.save_table("summary", summary)
display(evals.unstack("retriever"))

### 4.1 Query drift

In [ ]:
def emb(xs): return enc.encode(xs, batch_size=128, normalize_embeddings=True, show_progress_bar=False)
orig = emb([it["claim"] for it in items])
drift = {}
for stratum, key in STRATA.items():
    C = [it[key] for it in items]
    qs = {"A": C, "B": [f"{it['question']} {c}" for it, c in zip(items, C)], "C": [f"{it['question']} {c} {it['answer']}" for it, c in zip(items, C)],
          "F": [it[f"F_{key}"] for it in items]}
    for s, q in qs.items():
        drift[(stratum, s)] = round(float(np.mean(1 - np.sum(emb(q) * orig, axis=1))), 4)
drift_df = pd.Series(drift).unstack(0)
exp.log_result("query_drift", drift_df.to_dict()); display(drift_df)

## 5. Evaluation: H3a, H3b, selection

In [ ]:
def diff(stratum, retriever, a, b, metric, split="evaluation"):
    X = R[(R.split == split) & (R.stratum == stratum) & (R.retriever == retriever)]
    xa = X[X.strategy == a].set_index("claim_id")[metric]; xb = X[X.strategy == b].set_index("claim_id")[metric].loc[xa.index]
    return cluster_bootstrap_diff(xa, xb, pd.Series(xa.index, index=xa.index), n_boot=cfg["decision_rules"]["n_boot"], seed=SEED)
h3 = {"H3a": {}, "H3b": {}}
for r in cfg["retrievers"]:
    h3["H3a"][r] = {s: diff("context-dependent", r, s, "A", "recall@10") for s in ("B", "C", "D", "E", "F", "oracle")}
    h3["H3b"][r] = {m: {s: diff("self-contained", r, s, "A", m) for s in ("B", "C", "D", "E", "F")} for m in ("ndcg@10", "p@5")}
h3a_ok = all(h3["H3a"][r]["B"]["ci_low"] > 0 for r in cfg["retrievers"])
h3b_ok = all(h3["H3b"][r]["ndcg@10"]["B"]["ci_low"] <= 0 for r in cfg["retrievers"])
h3["verdict"] = {"H3a": "SUPPORTED" if h3a_ok else "NOT SUPPORTED", "H3b": "SUPPORTED" if h3b_ok else "NOT SUPPORTED"}
tune = R[R.split == "tuning"].groupby("strategy")["ndcg@10"].mean().drop("oracle", errors="ignore")
h3["selection"] = {"tuning_mean_ndcg@10": tune.round(4).to_dict(), "selected": tune.idxmax()}
exp.log_result("hypotheses", h3)
print(json.dumps(h3["verdict"]), "| selected:", h3["selection"]["selected"])
tab = pd.DataFrame([{"retriever": r, "comparison": f"{s} − A", **{k: round(v, 4) for k, v in h3["H3a"][r][s].items() if k != "n_clusters"}}
                    for r in cfg["retrievers"] for s in h3["H3a"][r]])
print("H3a (context-dependent, recall@10, evaluation split)"); display(tab)
tab_b = pd.DataFrame([{"retriever": r, "metric": m, "comparison": f"{s} − A", **{k: round(v, 4) for k, v in h3["H3b"][r][m][s].items() if k != "n_clusters"}}
                      for r in cfg["retrievers"] for m in ("ndcg@10", "p@5") for s in h3["H3b"][r][m]])
print("H3b (self-contained, evaluation split)"); display(tab_b)
exp.save_table("h3a", tab); exp.save_table("h3b", tab_b)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
order = ["A", "B", "C", "D", "E", "F", "oracle"]
for ax, stratum in zip(axes, STRATA):
    d = evals.xs(stratum, level="stratum")["ndcg@10"].unstack("retriever").reindex([o for o in order if (stratum == "context-dependent" or o != "oracle")])
    d.plot.bar(ax=ax, rot=0); ax.set_title(f"{stratum} (evaluation split)"); ax.set_ylabel("nDCG@10"); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); exp.save_figure("strategies", fig); plt.show()

## 6. Error analysis

In [ ]:
X = R[(R.split == "evaluation") & (R.stratum == "context-dependent") & (R.retriever == "dense")].pivot(index="claim_id", columns="strategy", values="recall@10")
by_id = {it["claim_id"]: it for it in items}
lost = X[(X["oracle"] > 0) & (X["A"] == 0)].index[:4]
hurt = X[(X["A"] > 0) & (X["B"] < X["A"])].index[:4]
print("Dense, context-dependent: oracle finds evidence, claim-only (A) does not:")
display(pd.DataFrame([{"dependent": by_id[i]["dependent"], "question": by_id[i]["question"], "F": by_id[i]["F_dependent"]} for i in lost]))
print("Adding the question (B) loses evidence that A found:")
display(pd.DataFrame([{"dependent": by_id[i]["dependent"], "question": by_id[i]["question"]} for i in hurt]))

## 7. Real claims (NB03): how many are context-dependent, and what F does to them
Tagger: unresolved referent (pronoun or "the drug/condition/patients…" with no key term), or a key domain term (in-scope drug name or condition word) that appears in the question but not in the claim. **Spot check:** 40 tags inspected by the AI developer (not a human annotation); reported if the annotation file exists, otherwise NOT EVALUATED.

In [ ]:
resp = {json.loads(l)["response_id"]: json.loads(l) for l in (PROC / "responses" / "responses.jsonl").open(encoding="utf-8")}
real = [json.loads(l) for l in (PROC / "responses" / "claims.jsonl").open(encoding="utf-8")]
if SMOKE:
    real = real[:200]
concepts = [json.loads(l) for l in (PROC / "concepts.jsonl").open(encoding="utf-8")]
KEYS = domain_regex({n.lower() for c in concepts if c["source"] == "rxnorm" for n in [c["name"], *c["synonyms"]] if len(n) >= 4})
for c in real:
    c["context_dependent"], c["cd_reason"] = context_dependent(c["text"], resp[c["response_id"]]["question"], KEYS)
rc = pd.DataFrame(real)
prev = {"claims": len(rc), "context_dependent": int(rc.context_dependent.sum()), "share": round(float(rc.context_dependent.mean()), 3),
        "by_reason": rc[rc.context_dependent].cd_reason.value_counts().to_dict(),
        "share_by_generator": rc.groupby("generator").context_dependent.mean().round(3).to_dict()}
# tagger check by construction on SciFact (referent rule only; key terms there are not domain terms)
prev["referent_rule_on_synthetic_dependent"] = round(float(np.mean([bool(_REFERENT.search(it["dependent"])) for it in items])), 3)
prev["referent_rule_on_original_scifact"] = round(float(np.mean([bool(_REFERENT.search(it["claim"])) for it in items])), 3)
exp.log_result("real_claims_context_dependence", prev); print(json.dumps(prev, indent=1))

In [ ]:
rng = random.Random(SEED)
cd_idx = [i for i, c in enumerate(real) if c["context_dependent"]]; sc_idx = [i for i, c in enumerate(real) if not c["context_dependent"]]
k = cfg["real_claims"]["tagger_spot_check"] // 2
spot = [real[i] | {"question": resp[real[i]["response_id"]]["question"]} for i in sorted(rng.sample(cd_idx, min(k, len(cd_idx))) + rng.sample(sc_idx, min(k, len(sc_idx))))]
spot_file = ANN / ("nb04_tagger_spot_check_smoke.json" if SMOKE else "nb04_tagger_spot_check.json")
if spot_file.exists():
    ann = {a["claim_id"]: a for a in json.loads(spot_file.read_text(encoding="utf-8"))["items"]}
    ok = [ann[s["claim_id"]]["context_dependent"] == s["context_dependent"] for s in spot if s["claim_id"] in ann]
    tp = sum(1 for s in spot if s["claim_id"] in ann and s["context_dependent"] and ann[s["claim_id"]]["context_dependent"])
    spot_res = {"n": len(ok), "agreement": round(float(np.mean(ok)), 3), "precision_of_dependent_tag": f"{tp}/{sum(1 for s in spot if s['context_dependent'])}",
                "missed_dependent_among_self_contained_tags": sum(1 for s in spot if s["claim_id"] in ann and not s["context_dependent"] and ann[s["claim_id"]]["context_dependent"])}
else:
    (ANN / ("nb04_tagger_spot_check_TODO_smoke.json" if SMOKE else "nb04_tagger_spot_check_TODO.json")).write_text(json.dumps([{"claim_id": s["claim_id"], "question": s["question"], "claim": s["text"],
        "tag": s["context_dependent"], "reason": s["cd_reason"]} for s in spot], ensure_ascii=False, indent=1), encoding="utf-8")
    spot_res = "NOT EVALUATED (sample written for annotation)"
exp.log_result("tagger_spot_check", spot_res); print(spot_res)

In [ ]:
controls = rng.sample(sc_idx, min(cfg["real_claims"]["self_contained_controls"], len(sc_idx)))
targets = sorted(set(cd_idx) | set(controls))
with managed() as mc:
    for i in targets:
        c = real[i]; rsp = resp[c["response_id"]]
        out, _ = decontextualize(mc, c["text"], rsp["question"], rsp["answer"], seed=SEED, tag=f"dectx:real:{c['claim_id']}")
        c["standalone_text"] = out
norm = lambda t: re.sub(r"\W+", " ", (t or "").lower()).strip()
dep = [real[i] for i in cd_idx]; ctl = [real[i] for i in controls]
real_f = {"rewritten_dependent": len(dep), "dependent_changed": round(float(np.mean([norm(c["standalone_text"]) != norm(c["text"]) for c in dep])), 3),
          "dependent_now_tagged_self_contained": round(float(np.mean([not context_dependent(c["standalone_text"] or c["text"], resp[c["response_id"]]["question"], KEYS)[0] for c in dep])), 3),
          "controls": len(ctl), "controls_unchanged": round(float(np.mean([norm(c["standalone_text"]) == norm(c["text"]) for c in ctl])), 3),
          "length_ratio_dependent": round(float(np.mean([len((c["standalone_text"] or "").split()) / max(1, len(c["text"].split())) for c in dep])), 2)}
exp.log_result("real_claims_rewrite_F", real_f); print(json.dumps(real_f, indent=1))
display(pd.DataFrame([{"question": resp[c["response_id"]]["question"][:90], "claim": c["text"], "F": c["standalone_text"]} for c in dep[:8]]))
with (PROC / "responses" / ("claims_contextualized_smoke.jsonl" if SMOKE else "claims_contextualized.jsonl")).open("w", encoding="utf-8") as fh:
    for c in real:
        fh.write(json.dumps({**c, "standalone_text": c.get("standalone_text") or c["text"],
                             "standalone_source": "F" if c.get("standalone_text") else "original"}, ensure_ascii=False) + "\n")
exp.finish(); print("results:", exp.dir)

## 8. Discussion
All figures are copied from the outputs above (`artifacts/experiments/EXP-04-001/`).

### 8.1 Test items (synthetic, by construction)
- 693 SciFact claims → **476 valid items** (68.7%): 155 tuning / 321 evaluation.
- Rejected by the rules: the subject still named in the dependent claim (74), not one sentence of similar length (58), content added (38), entity not in the question (27), entity not in the claim (16), other (4).
- The rules removed exactly the failure modes seen in the probe (invented sentences, questions restating the finding).

### 8.2 Hypotheses (evaluation split, 321 claims)
| | Retriever | Claim only (A) | Question + claim (B) | Difference [95% CI] |
|---|---|---|---|---|
| **H3a** context-dependent, Recall@10 | BM25 | 0.759 | 0.906 | **+0.146 [0.107, 0.185]** |
| | Dense | 0.721 | 0.967 | **+0.246 [0.202, 0.293]** |
| **H3b** self-contained, nDCG@10 | BM25 | 0.835 | 0.820 | −0.015 [−0.033, 0.001] |
| | Dense | 0.901 | 0.890 | **−0.011 [−0.022, −0.000]** |

- **H3a: SUPPORTED.** When a claim lost its subject, adding the question recovers most of the lost recall for both retrievers.
- **H3b: SUPPORTED.** For self-contained claims, adding the question does not help, and with the dense retriever it slightly but significantly **hurts** (P@5 −0.003 [−0.005, −0.001]).
- Separate retrieval + merge (D) is clearly worse on self-contained claims (nDCG@10 0.742 vs 0.901, dense). Rank fusion lets the question's documents displace the claim's.

### 8.3 Strategy selection: F (LLM rewrite), chosen on the tuning split
| Strategy | Tuning nDCG@10 (mean, both strata, both retrievers) |
|---|---|
| F rewrite | **0.868** |
| C question + claim + answer | 0.863 |
| B question + claim | 0.859 |
| E weighted fusion (w = 0.6) | 0.844 |
| D rank fusion | 0.779 |
| A claim only | 0.734 |

- On the evaluation split, F **matches the oracle** on context-dependent claims (dense nDCG@10 0.904 vs oracle 0.901; BM25 0.835 vs 0.835).
- F also does not degrade self-contained claims (0.906 vs 0.901 dense), unlike B.
- Query drift explains why: the F queries stay closest to the original claim (drift 0.004 vs 0.045 for B and 0.092 for C).
- On SciFact, F restored the removed subject in **98.3%** of dependent claims and returned 82.4% of already-standalone claims unchanged (the rest were minor rewordings).

### 8.4 Real claims (NB03 output, no relevance labels)
- **42.7% (1,185 / 2,777) are tagged context-dependent**: 701 because the key entity is only in the question, 484 for an unresolved referent. The rate is the same for both generators (43.5% / 42.0%).
- **Tagger spot check (40 claims, AI-annotated): 80% agreement.**
  - 16/20 "dependent" tags are correct. The false positives are complete statements in which the question happened to name a drug.
  - 4/20 "self-contained" tags are wrong. All four have an unresolved reference *plus* a domain word ("**These factors** increase LDL cholesterol"), which suppresses the referent rule. So the 42.7% is approximate, with errors in both directions.
- **F on real claims is far less complete than on synthetic ones.**
  - It changed 60.6% of the 1,185 dependent claims, and 50.8% are tagged self-contained afterwards (vs 98.3% restoration on SciFact).
  - The unchanged ones are largely generic advice ("Always ensure that you are using a genuine product…"), plus tagger false positives.
  - When F rewrites, it typically restores the drug or condition ("stopping the medication" → "stopping simvastatin"). Occasionally it turns a list fragment into a stated relation ("9. Headache" → "Headache is a sign of low blood pressure"): correct in context, but it introduces a predicate the answer only implied.
  - 83% of the self-contained controls came back unchanged; mean length grows by 21% for dependent claims.

## 9. Limitations
- **Synthetic, favourable construction:** each dependent claim lacks exactly one entity, and the question names exactly that entity in a formulaic way ("What do we know about X?"). Real dependence is messier (missing populations, conditions, list context), and the real-claim analysis shows F is much less complete there. The H3 effect sizes are therefore an **upper bound** on real-world gains.
- **Benchmark domain:** SciFact is general biomedicine, with claims written from citation contexts, not patient-facing answers.
- **Same model generates and rewrites:** Qwen3-8B wrote the synthetic context and also performs F. It may undo its own phrasing more easily than real phrasing.
- **No retrieval labels for real claims:** retrieval quality of F on real claims is not measured, only how often it rewrites and whether the rewrite still looks context-dependent.
- **Tagger:** precision and recall are only spot-checked (40 items, AI annotation; D5).
- **Fixed small reference retrievers:** NB05 re-checks the strategy with the final retrieval stack (NB08 re-runs winners).

## 10. Conclusion and decision
**Conclusion:**
- Context matters, but only for claims that need it.
- Adding the question rescues context-dependent claims (+15 to +25 Recall@10 points) and slightly hurts self-contained ones.
- An LLM rewrite into a standalone claim (F) gets the benefit without the cost on the synthetic benchmark, matching the oracle.
- On real answers, 43% of claims need context, and F resolves roughly half of them.

**H3a: SUPPORTED. H3b: SUPPORTED** (pre-registered rules; synthetic test items).

**Decision (proposed, pending owner approval; ADR-0011):**
- Use **strategy F** for every claim: retrieve with the standalone rewrite (`standalone_text` in `data/processed/2026-11-20/responses/claims_contextualized.jsonl`).
- Keep the original claim text for verification and display.
- Do **not** prepend the question by default (it degrades self-contained claims).
- For claims still tagged context-dependent after F, NB05 tests question + claim (B) as a fallback channel.

**Next steps (NB05, retrieval):**
- Sparse, dense, graph and fusion channels plus re-ranking over the CliniVer chunked corpus and SciFact.
- Re-check the chunk-size and whole-abstract questions left open by NB02.
- Test F vs F + B fallback on real claims where labels can be obtained.